# Structured Output

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader,PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma,FAISS
import fitz  

#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os
import numpy as np
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.document_loaders import WikipediaLoader
from PIL import Image
import torch
from transformers import CLIPProcessor,CLIPModel
import base64
import io
from langchain.messages import SystemMessage,HumanMessage,AIMessage
from pydantic import BaseModel,Field
from langchain.agents import create_agent

C:\Users\kanha\AppData\Local\Temp\ipykernel_28016\1312583015.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader,PyMuPDFLoader


In [2]:
## Step 4: LLM and prompt
import os
from dotenv import load_dotenv

load_dotenv()
API=os.getenv("OPENAI_API_KEY")
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key=API)
response=llm.invoke("Why do parrots talk?")
response

AIMessage(content="Parrots are renowned for their ability to mimic human speech and other sounds, but why do they engage in this behavior? The reasons are complex and multifaceted, involving a combination of evolutionary, social, and cognitive factors. Here are some theories:\n\n1. **Communication and Social Bonding**: In the wild, parrots use vocalizations to communicate with their flock members, warning them of predators, signaling food sources, and maintaining social bonds. Talking may be an extension of this natural behavior, allowing them to bond with their human caregivers and interact with their environment.\n2. **Mimicry and Learning**: Parrots are intelligent birds with a strong desire to learn and mimic sounds they hear. By replicating human speech, they may be attempting to understand and participate in the communication process. This mimicry can also serve as a form of play and entertainment.\n3. **Attention and Reward**: Parrots quickly learn that talking can elicit a resp

In [3]:
from langchain.messages import SystemMessage,HumanMessage,AIMessage

## Pydantic

In [5]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description='The title of the movie')
    year:int=Field(description="This year of movie released")
    director:str=Field(desscription="The director of the movie")
    rating:float=Field(description='The movies rating out of 10')


C:\Users\kanha\AppData\Local\Temp\ipykernel_28016\3025300310.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desscription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  director:str=Field(desscription="The director of the movie")


In [6]:
model_with_structure=llm.with_structured_output(Movie)

In [7]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000204A21F0490>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000204A2297490>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': 

In [10]:
model_with_structure.invoke('provide me details of movie conjuring')

Movie(title='The Conjuring', year=2013, director='James Wan', rating=7.9)

# Message ouptut alongside parsed structure

In [11]:
model_with_structure=llm.with_structured_output(Movie,include_raw=True)

In [12]:
model_with_structure.invoke('provide me details of movie conjuring')

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 's3rjvcgw2', 'function': {'arguments': '{"director":"James Wan","rating":7.9,"title":"The Conjuring","year":2013}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 279, 'total_tokens': 312, 'completion_time': 0.078534685, 'completion_tokens_details': None, 'prompt_time': 0.019203342, 'prompt_tokens_details': None, 'queue_time': 0.057250481, 'total_time': 0.097738027}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8edc-f3cf-72b0-88ca-88787e3840c3-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'James Wan', 'rating': 7.9, 'title': 'The Conjuring', 'year': 2013}, 'id': 's3rjvcgw2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 279, 'output_tokens': 33, 'total_

## nested structure

In [14]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name:str
    role:str
class Moviedetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float|None=Field(None,description='budget in millions USD')


In [16]:
model_with_structure=llm.with_structured_output(Moviedetails)

In [17]:
model_with_structure.invoke('provide me details of movie conjuring')

Moviedetails(title='The Conjuring', year='2013', cast=[Actor(name='Vera Farmiga', role='Lorraine Warren'), Actor(name='Patrick Wilson', role='Ed Warren')], genres=['Horror', 'Mystery', 'Thriller'], budget=20.0)

## Typed Dict

In [19]:
from typing import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str, "The title of the movie"]
    year: Annotated[str, "The year"]

In [21]:
model_with_structure=llm.with_structured_output(MovieDict)
model_with_structure.invoke('provide me details of movie conjuring')

{'title': 'The Conjuring', 'year': '2013'}

In [22]:
llm.profile

{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

## Data Classes

In [23]:
os.environ["GROQ_API_KEY"] =os.getenv("OPENAI_API_KEY")

In [26]:
from langchain.agents import create_agent
class Contract(BaseModel):
    name:str=Field(description="The name of person")
    email:str=Field(description="the email")
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=Contract
 
)
result=agent.invoke({
    "messages":[{"role":"user","content":"Extract the contact info:John Doe,k@gmail.com"}]
})
print(result['structured_response'])

name='John Doe' email='k@gmail.com'


In [27]:
result

{'messages': [HumanMessage(content='Extract the contact info:John Doe,k@gmail.com', additional_kwargs={}, response_metadata={}, id='251d56e9-b5fa-4269-aee6-8f4cc9312e4e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'k5yrmdfr3', 'function': {'arguments': '{"email":"k@gmail.com","name":"John Doe"}', 'name': 'Contract'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 242, 'total_tokens': 262, 'completion_time': 0.040142662, 'completion_tokens_details': None, 'prompt_time': 0.078125861, 'prompt_tokens_details': None, 'queue_time': 0.154154286, 'total_time': 0.118268523}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8eea-6a55-74d2-9670-0eae625b8d29-0', tool_calls=[{'name': 'Contract', 'args': {'email': 'k@gmail.com', 'name': 'John Doe'}, 'id': 'k5yrmdfr3', 'type':

In [31]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class Contact:
    name: str

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=Contact,
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract the contact info: John Doe",
            }
        ]
    }
)

print(result["structured_response"])

Contact(name='John Doe')
